## Gaussian Process Regression

# Gaussian Process Regression for Building Energy Analysis

## 1. Theoretical Overview

Gaussian Process Regression (GPR) is a Bayesian approach to regression where we treat the underlying function as a random variable. Unlike traditional parametric regression, which searches for a single "best" set of weights (like in a neural network), GPR is non-parametric and maintains a distribution over all possible functions that could fit the data.

The problem can be broken down into three conceptual layers:

### The Latent Process (The "Signal")
We assume there exists an underlying, "clean" process $X_g$ (where $g$ represents the input dimensions, such as our building parameters $X_1 \dots X_8$). We define this process as a Gaussian Process. For any finite set of points, the values of $X_g$ follow a Multivariate Gaussian distribution.

This process is entirely defined by two components:
* **Mean Function** $\mu_g$: Represents the expected average behavior of the function.
* **Kernel (Covariance Function)** $\kappa(g,g')$: Defines the "shape," smoothness, and correlation of the functions based on the distance between data points.

### The Noisy Observation (The "Measurement")
In the real world (or in complex simulations like Ecotect), we rarely capture the clean signal $X_g$ perfectly. Instead, we observe $Y_g$, which is the true signal corrupted by random noise $\nu_g$.

* **The Problem:** We have a collection of $n$ noisy data points $Y_n = [y_{g1}, y_{g2}, \dots, y_{gn}]$.
* **The Objective:** Given these noisy observations, we must reconstruct the most likely version of the clean signal $X_g$ and determine how much we should trust that reconstruction.

### Summary of Optimal Inference
The goal of Gaussian Process Regression is to take discrete, noisy observations and produce a continuous, probabilistic function. It doesn't just output a single predictive line; it provides a "corridor of probability" (confidence intervals). This tells us not only our predicted value but also our level of uncertainty, which naturally increases as we move further away from our known data points.

**In short:** GPR uses Bayesian inference to squash an infinite space of possible functions into a narrow band of likely functions that honor our noisy data.

---

## 2. Discussion: Modeling Heating and Cooling Loads

In this dataset, we are analyzing the energy efficiency of 12 building shapes resulting in eight physical attributes ($X_1$ to $X_8$, representing features like glazing area, orientation, and surface area) to predict two distinct responses: Heating Load ($Y_1$) and Cooling Load ($Y_2$).

Exploring the possibility of modeling these targets as a **single parameter Gaussian process** (i.e., training one standard, single-output GP for Heating Load and another separate single-output GP for Cooling Load) yields several important conclusions:

### Advantages of the Single Parameter GP Approach
* **Handling Non-Linearity:** Energy loads have highly complex, non-linear relationships with building geometry (e.g., the relationship between glazing distribution and cooling load). GPR’s kernel trick perfectly captures these non-linearities without the need for manual feature engineering.
* **Uncertainty Quantification:** Because simulations like Ecotect often have inherent assumptions and simplifications, the predictive variance (the "corridor of probability") provided by GPR is incredibly valuable. It tells architects and engineers exactly how confident the model is in its load predictions for a novel building design.
* **Data Efficiency:** GPR performs exceptionally well on small to medium-sized datasets. Given that the dataset originates from variations of only 12 base building shapes, GPR is less prone to overfitting compared to data-hungry models like Deep Neural Networks.

### Limitations and the Case for Multi-Output GP
While modeling $Y_1$ and $Y_2$ as independent, single parameter GPs is highly effective, it introduces a physical blind spot. Heating and cooling loads in a building are inherently correlated (e.g., increasing the glazing area on a south-facing wall might simultaneously lower the heating load due to solar gain while drastically increasing the cooling load).

**Conclusion:** Modeling the loads as independent single parameter Gaussian processes is a robust, mathematically rigorous baseline that will yield highly accurate predictions and excellent uncertainty bounds. However, because $Y_1$ and $Y_2$ are thermodynamically linked, an advanced **Multi-Output Gaussian Process (MOGP)**—which calculates a cross-covariance between the two responses—would theoretically capture the underlying physics even better by sharing information between the heating and cooling tasks.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, WhiteKernel
from sklearn.model_selection import train_test_split

# 1. Generate Synthetic "Ecotect" Data (Placeholder for your real dataset)
# 8 Features (X1-X8), 2 Responses (Heating Y1, Cooling Y2)
np.random.seed(42)
num_samples = 150 # Assuming variations of the 12 base shapes

# Features: e.g., Glazing Area, Orientation, Surface Area, etc.
X = np.random.rand(num_samples, 8)

# Simulating Heating Load (Y1) with some non-linear relationships to X
y_heating = 10 + 15 * X[:, 0]**2 + 5 * np.sin(X[:, 1] * np.pi) + np.random.normal(0, 1, num_samples)

# 2. Train-Test Split to evaluate our predictions on unseen building variations
X_train, X_test, y_train, y_test = train_test_split(X, y_heating, test_size=0.2, random_state=42)

# 3. Define the Kernel
# Constant (scale) * RBF (smoothness) + WhiteKernel (handles the noisy observations)
kernel = C(1.0, (1e-3, 1e3)) * RBF(length_scale=np.ones(8), length_scale_bounds=(1e-2, 1e2)) + WhiteKernel(noise_level=1, noise_level_bounds=(1e-3, 1e1))

# 4. GP Regression & Fitting for Heating Load (Single Parameter approach)
gp_heating = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, normalize_y=True)
gp_heating.fit(X_train, y_train)

# 5. Predict on the test set to get Means and Standard Deviations (Uncertainty)
y_pred, y_std = gp_heating.predict(X_test, return_std=True)

# 6. Visualize High-Dimensional GP with Plotly (Actual vs. Predicted with Error Bars)
fig = go.Figure()

# Add ideal 1:1 line (Perfect prediction)
min_val, max_val = min(y_test), max(y_test)
fig.add_trace(go.Scatter(
    x=[min_val, max_val], y=[min_val, max_val],
    mode='lines', line=dict(color='gray', dash='dash'),
    name='Perfect Prediction'
))

# Add Predictions with 95% Confidence Intervals as Error Bars
fig.add_trace(go.Scatter(
    x=y_test, y=y_pred,
    mode='markers',
    error_y=dict(
        type='data', array=1.96 * y_std, visible=True,
        color='rgba(0,100,80,0.4)', thickness=1.5
    ),
    marker=dict(color='rgb(0,100,80)', size=8, line=dict(color='white', width=1)),
    name='Predictions ± 95% CI'
))

# Formatting
fig.update_layout(
    title="GPR: Actual vs. Predicted Heating Load (8D Input Space)",
    xaxis_title="Actual Heating Load (Ecotect Simulation)",
    yaxis_title="Predicted Heating Load (GP Mean)",
    template="plotly_white",
    hovermode="closest"
)

fig.show()

# Linear Regression

# Linear Regression for Green Building Energy Demand

## 1. Theoretical Overview

Linear Regression is a foundational, parametric statistical approach. Unlike non-parametric models (like Gaussian Processes) that maintain a distribution over possible functions, Linear Regression seeks a single, deterministic equation. It assumes that the target variable (in this case, `predicted_energy_demand`) can be accurately modeled as a weighted linear combination of the input features.

The conceptual layers of this approach are:

### The Linear Hypothesis (The "Signal")
We assume there is a linear underlying relationship between the building's environmental features ($X$) and its energy demand ($Y$). The model is defined by the equation:

$$Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \beta_p X_p + \epsilon$$

Where:
* $Y$ is the `predicted_energy_demand`.
* $X_1 \dots X_p$ are our selected data parameters (features).
* $\beta_0$ is the intercept (baseline energy demand when all inputs are zero).
* $\beta_1 \dots \beta_p$ are the learned weights (coefficients) indicating the impact of each feature.
* $\epsilon$ represents the irreducible error (noise).

### The Optimization (The "Fit")
Given your 2,400 samples, the algorithm uses Ordinary Least Squares (OLS) to find the specific set of $\beta$ weights that minimizes the sum of the squared differences between the actual energy demand and the model's predicted demand.

---

## 2. Justification of Parameter Choice

To build a highly effective linear model, we cannot simply throw all available multi-source data at the algorithm. We must select a "suitable set" of parameters that capture the core thermodynamics while avoiding **multicollinearity** (where features are too highly correlated with each other, confusing the model).

A justified feature set for this dataset would include:

* **Temperature Differential ($\Delta T$):** Instead of using raw indoor and outdoor temperatures separately, calculating the difference between the two is highly effective. The energy required to heat or cool a space scales linearly with this differential.
* **Solar Irradiance / Light Levels:** Direct solar radiation is a massive contributor to heat gain. In green buildings, solar exposure has a near-linear relationship with increased cooling loads.
* **Occupancy Count:** Humans emit sensible and latent heat, and require fresh ventilation. The relationship between the number of people in a room and base energy consumption is strongly linear.
* **Humidity / Moisture (Latent Load):** HVAC systems use significant energy to dehumidify air. Including relative humidity is crucial because latent heat removal is often a hidden energy sink.

---

## 3. Discussion & Conclusions

Exploring the possibility of predicting `predicted_energy_demand` purely through linear regression yields the following insights:

### Advantages of the Linear Approach
* **Absolute Interpretability:** The greatest strength of this model is its transparency. If the coefficient for `Occupancy_Count` is 15.2, we know that every additional person adds exactly 15.2 units to the energy demand. This allows green building architects to make immediate, understandable design decisions.
* **Baseline Efficiency:** With 2,400 samples, a linear model will train in milliseconds and serves as the perfect baseline to benchmark more complex Deep Learning models against.

### Limitations & The Non-Linearity Problem
* **Thermal Dynamics are Complex:** Real-world building physics involves heavy non-linearities. For instance, the concept of "thermal mass" means a building absorbs heat slowly and releases it hours later. A basic linear model struggles to capture these time-lagged, non-linear effects.
* **Interaction Effects:** Linear regression assumes features act independently. However, high solar irradiance combined with high occupancy requires exponentially more cooling than the sum of their individual effects.

**Final Conclusion:** Modeling the `predicted_energy_demand` as a linear relationship is a highly valuable first step that provides crucial, interpretable insights into the primary drivers of energy use. However, because multi-source environmental data contains complex, interacting thermodynamic variables, the linear model will likely hit a performance ceiling (showing patterns in its residual errors). To achieve the predictive accuracy required for advanced "energy efficiency optimization," this linear baseline would eventually need to be upgraded with interaction terms, regularization (Ridge/Lasso), or transitioned to a non-linear algorithm.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

# 1. Generate Synthetic Multi-Source Data (Placeholder for your kagglehub df2)
np.random.seed(42)
n_samples = 2400

# Generating the justified parameters
df = pd.DataFrame({
    'Temp_Differential': np.random.uniform(0, 25, n_samples),  # delta T in Celsius
    'Solar_Irradiance': np.random.uniform(0, 1000, n_samples), # W/m^2
    'Occupancy_Count': np.random.randint(0, 50, n_samples),    # Number of people
    'Humidity_Percent': np.random.uniform(30, 90, n_samples)   # Relative humidity
})

# Simulating true linear relationship with some random noise (epsilon)
# True weights: dT: 2.5, Solar: 0.05, Occ: 3.0, Hum: 0.8
df['Predicted_Energy_Demand'] = (
    15.0 + # Intercept
    2.5 * df['Temp_Differential'] +
    0.05 * df['Solar_Irradiance'] +
    3.0 * df['Occupancy_Count'] +
    0.8 * df['Humidity_Percent'] +
    np.random.normal(0, 10, n_samples) # Noise
)

# 2. Prepare Data
X = df[['Temp_Differential', 'Solar_Irradiance', 'Occupancy_Count', 'Humidity_Percent']]
y = df['Predicted_Energy_Demand']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train the Linear Regression Model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# 4. Evaluate
y_pred = lr_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

# 5. Extract Coefficients for Interpretability
coefficients = lr_model.coef_
features = X.columns

# 6. Visualize with Plotly (Dual Plot: Coefficients and Model Fit)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Feature Impact (Weights)", f"Model Fit (R² = {r2:.2f})"))

# Subplot 1: The Coefficients (Interpretability)
fig.add_trace(go.Bar(
    x=features, y=coefficients,
    marker_color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728'],
    name="Beta Weights"
), row=1, col=1)

# Subplot 2: Actual vs Predicted
fig.add_trace(go.Scatter(
    x=y_test, y=y_pred, mode='markers',
    marker=dict(color='rgba(31, 119, 180, 0.5)', size=6),
    name="Test Set Predictions"
), row=1, col=2)

# Perfect fit line
fig.add_trace(go.Scatter(
    x=[min(y_test), max(y_test)], y=[min(y_test), max(y_test)],
    mode='lines', line=dict(color='black', dash='dash'),
    name="Ideal Fit"
), row=1, col=2)

# Formatting
fig.update_layout(
    title="Linear Regression Analysis: Green Building Energy Demand",
    template="plotly_white",
    showlegend=False,
    height=500
)

fig.update_yaxes(title_text="Weight (Impact on Energy)", row=1, col=1)
fig.update_xaxes(title_text="Actual Energy Demand", row=1, col=2)
fig.update_yaxes(title_text="Predicted Energy Demand", row=1, col=2)

fig.show()